In [1]:
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import warnings
import json

warnings.filterwarnings('ignore')
plt.rcParams["font.sans-serif"] = ["SimHei"] # 正常显示中文
plt.rcParams["axes.unicode_minus"] = False # 正常显示负号

DEFAULT_PATH = Path('D:/Python/dachuang2026')# 显示项目路径
AMINO_ACIDS = 'ARNDCEQGHILKMFPSTWYV'
ALL_DIPEPTIDES = [a + b for a in AMINO_ACIDS for b in AMINO_ACIDS]# 400种二肽组合

In [2]:
def read_fasta(fname):
    """读取FASTA文件并转化为DataFrame"""
    records = []
    for rec in SeqIO.parse(fname, "fasta"):
        records.append((rec.id, str(rec.seq).upper()))
        # .upper()确保序列大写，避免后续处理中的大小写问题
    return pd.DataFrame(records, columns=["Id", "Sequence"])

In [3]:
def get_dipeptide_freq_dict(seq):
    """高效计算二肽频率"""
    if len(seq) < 2:
        return {}
    dipeptides = [seq[i:i + 2] for i in range(len(seq) - 1)]
    total = len(dipeptides)
    return {k: v / total for k, v in Counter(dipeptides).items()}

In [4]:
def get_termini_composition(seq):
    """提取末端5个氨基酸的组成特征"""
    features = {}
    n_term = seq[:5] if len(seq) >= 5 else seq
    c_term = seq[-5:] if len(seq) >= 5 else seq

    for aa in AMINO_ACIDS:
        features[f'Nterm_{aa}'] = n_term.count(aa) / len(n_term) if n_term else 0
        features[f'Cterm_{aa}'] = c_term.count(aa) / len(c_term) if c_term else 0
    return features

In [5]:
dic = read_fasta(DEFAULT_PATH / 'data/test2.fasta')
dic

,Id,Sequence
0,sequence_1|1,MLKFILALALFLHLTMEASAACKGKKCPPTGFVGMRG
1,sequence_2|1,RPPGFTPFRKA
2,sequence_3|1,GDCHKFLGWCRGEPDPCCEHLSCSRKHGWCVWDWTV
3,sequence_4|1,KAKCADIDQPCKTSCDCCETKGACTCYKSGCVCRMGSFATCKK
4,sequence_5|1,SQCCYQICSPC
...,...,...
577,sequence_578|0,KGKGFWSWASKATSWLTGPQQPGSPLLKKHR
578,sequence_579|0,MKVRNSLRSAKARDKNCRVVRRHGRVYVINKKNPRMKARQG
579,sequence_580|0,DVPSANANANNQRTAAAKPQANAEASS
580,sequence_581|0,MKKTDKILKEIGIQRVAIMEGKKYSKGFMEDGDIGVQY


In [6]:
def process_data(dic, feature_config=None):
    """
    针对肽毒性预测优化的特征工程（含二级结构语法修正与两亲性特征）
    :param dic: 原始 DataFrame，需包含 'Id' (可选) 和 'Sequence' 列
    :param feature_config: 字典类型，用于预测模式下对齐特征列
    """
    # 1. 标签提取与清理
    if 'Id' in dic.columns:
        dic['toxicity'] = dic['Id'].apply(lambda x: x.split('|')[1] if '|' in x else '0')
        dic = dic.drop(columns=['Id'])

    # 确保序列为大写，防止计算出错
    dic['Sequence'] = dic['Sequence'].str.upper()

    # 2. 基础物理化学特征
    dic['length'] = dic['Sequence'].apply(len)

    # 质量计算优化：使用字典映射提升速度
    df_aa = pd.read_csv(DEFAULT_PATH / 'data/amino_acids.csv').set_index('Called')
    mass_dict = df_aa['Mass'].to_dict()
    hydrone_mass = 18.01528

    dic['mass'] = dic['Sequence'].apply(
        lambda seq: sum(mass_dict.get(aa, 0) for aa in seq) - (len(seq) - 1) * hydrone_mass
    )

    # 3. 创建 ProteinAnalysis 对象缓存（增加空序列检查）
    analyzers = dic['Sequence'].apply(lambda x: ProteinAnalysis(x) if len(x) > 0 else None)

    # 4. 理化指标计算
    dic['hydrophobicity'] = analyzers.apply(lambda x: x.gravy() if x else 0)
    dic['isoelectric_point'] = analyzers.apply(lambda x: x.isoelectric_point() if x else 7.0)
    dic['charge_at_pH7.4'] = analyzers.apply(lambda x: x.charge_at_pH(7.4) if x else 0.0)

    # 5. 二级结构特征（语法修正版）
    # 获取 (helix, turn, sheet) 元组列表
    ss_results = analyzers.apply(lambda x: x.secondary_structure_fraction() if x else (0.0, 0.0, 0.0))
    # 转换为 DataFrame 并合并，确保索引对齐
    ss_df = pd.DataFrame(ss_results.tolist(), columns=['helix', 'turn', 'sheet'], index=dic.index)
    dic = pd.concat([dic, ss_df], axis=1)

    # 6. 两亲性模拟特征 (新增)
    # 通过比较序列两端的疏水性差异来模拟两亲性
    dic['amphiphilicity_index'] = dic['Sequence'].apply(
        lambda seq: abs(ProteinAnalysis(seq[:len(seq)//2]).gravy() -
                        ProteinAnalysis(seq[len(seq)//2:]).gravy()) if len(seq) >= 4 else 0.0
    )

    # 7. 氨基酸组成频率 (AAC)
    for aa in AMINO_ACIDS:
        dic[f'comp_{aa}'] = dic['Sequence'].apply(lambda seq: seq.count(aa) / len(seq) if len(seq) > 0 else 0)

    # 8. 二肽频率 (DPC) 与特征对齐逻辑
    all_freqs = [get_dipeptide_freq_dict(seq) for seq in dic['Sequence']]

    if feature_config is None:
        # 训练模式：计算方差并筛选前 50 个特征
        dipeptide_variance = {}
        for dip in ALL_DIPEPTIDES:
            vals = [f.get(dip, 0) for f in all_freqs]
            if np.var(vals) > 0:
                dipeptide_variance[dip] = np.var(vals)

        top_dips = sorted(dipeptide_variance.items(), key=lambda x: x[1], reverse=True)[:50]
        selected_dipeptides = [d[0] for d in top_dips]
        current_config = {'top_dipeptides': selected_dipeptides}
    else:
        # 预测模式：使用传入的配置，确保列名一致
        selected_dipeptides = feature_config.get('top_dipeptides', [])
        current_config = feature_config

    for dip in selected_dipeptides:
        dic[f'dip_{dip}'] = [f.get(dip, 0) for f in all_freqs]

    # 9. 序列复杂度与分组特征
    dic['seq_complexity'] = dic['Sequence'].apply(lambda seq: len(set(seq)) / len(seq) if len(seq) > 0 else 0)
    dic['charged_ratio'] = dic['Sequence'].apply(
        lambda seq: sum(seq.count(aa) for aa in 'KRHDE') / len(seq) if len(seq) > 0 else 0
    )

    # 10. 末端组成特征合并优化
    termini_features = dic['Sequence'].apply(get_termini_composition)
    termini_df = pd.DataFrame(termini_features.tolist(), index=dic.index)
    dic = pd.concat([dic, termini_df], axis=1)

    # 11. 毒性相关 Motif
    toxic_motifs = ['RR', 'KK', 'RK', 'KR', 'CXC', 'CC', 'RGD']
    for motif in toxic_motifs:
        dic[f'motif_{motif}'] = dic['Sequence'].apply(
            lambda seq: seq.count(motif) / max(1, len(seq) - len(motif) + 1)
        )

    # 输出统计信息
    print(f"处理完成！样本数: {len(dic)}")
    feature_cols = [col for col in dic.columns if col not in ['Sequence', 'toxicity']]
    print(f"特征维度: {len(feature_cols)}")

    return dic, current_config

In [7]:
processed_train, train_config = process_data(dic)
processed_train

处理完成！样本数: 582
特征维度: 128


,Sequence,toxicity,length,mass,hydrophobicity,isoelectric_point,charge_at_pH7.4,helix,turn,sheet,...,Cterm_Y,Nterm_V,Cterm_V,motif_RR,motif_KK,motif_RK,motif_KR,motif_CXC,motif_CC,motif_RGD
0,MLKFILALALFLHLTMEASAACKGKKCPPTGFVGMRG,1,37,3952.90632,0.697297,9.697931,3.263581,0.513514,0.189189,0.351351,...,0.0,0.0,0.2,0.000000,0.027778,0.000000,0.0,0.0,0.000000,0.0
1,RPPGFTPFRKA,1,11,1273.49190,-1.036364,11.999968,2.554897,0.181818,0.363636,0.272727,...,0.0,0.0,0.0,0.000000,0.000000,0.100000,0.0,0.0,0.000000,0.0
2,GDCHKFLGWCRGEPDPCCEHLSCSRKHGWCVWDWTV,1,36,4234.80510,-0.577778,6.269761,-1.481168,0.166667,0.305556,0.277778,...,0.0,0.0,0.2,0.000000,0.000000,0.028571,0.0,0.0,0.028571,0.0
3,KAKCADIDQPCKTSCDCCETKGACTCYKSGCVCRMGSFATCKK,1,43,4577.42774,-0.302326,8.585335,3.294810,0.302326,0.232558,0.186047,...,0.0,0.0,0.0,0.000000,0.023810,0.000000,0.0,0.0,0.023810,0.0
4,SQCCYQICSPC,1,11,1234.45340,0.272727,5.232223,-0.847291,0.000000,0.272727,0.181818,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.100000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
577,KGKGFWSWASKATSWLTGPQQPGSPLLKKHR,0,31,3465.97810,-0.970968,11.388421,5.581526,0.322581,0.354839,0.290323,...,0.0,0.0,0.0,0.000000,0.033333,0.000000,0.0,0.0,0.000000,0.0
578,MKVRNSLRSAKARDKNCRVVRRHGRVYVINKKNPRMKARQG,0,41,4878.79660,-1.278049,11.999968,13.279693,0.292683,0.243902,0.195122,...,0.0,0.2,0.0,0.025000,0.025000,0.000000,0.0,0.0,0.000000,0.0
579,DVPSANANANNQRTAAAKPQANAEASS,0,27,2668.75702,-0.955556,6.069745,-0.443511,0.407407,0.407407,0.074074,...,0.0,0.2,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0
580,MKKTDKILKEIGIQRVAIMEGKKYSKGFMEDGDIGVQY,0,38,4349.12534,-0.550000,9.100180,1.267015,0.394737,0.236842,0.315789,...,0.2,0.0,0.2,0.000000,0.054054,0.000000,0.0,0.0,0.000000,0.0


In [8]:
processed_train.to_csv(DEFAULT_PATH / 'data/processed_train_data_done.csv', index=False)
with open(DEFAULT_PATH / 'data/feature_config.json', 'w') as f:
    json.dump(train_config, f)

In [9]:
processed_train.shape

(582, 130)